In [ ]:
import os
import pandas as pd

from amendements_intelligents.loaders.plfss_json_loader import PLFSSJsonLoader
from amendements_intelligents.embeddings.expert_dbs import ExpertiseChromaDB

# Usage example
DATA_FOLDER = os.getenv("DATA_FOLDER")
sheet_name = "PLFSS 2024"
expertise_description_file_name = "clean_expertise_descriptions"

In [ ]:
from amendements_intelligents.utils.expertise_helpers import create_expertise_dict


file_path = f"{DATA_FOLDER}/{expertise_description_file_name}.csv"
description_col = "Sujets pris en charge"
corresponding_expert_col = "Personne en charge"

df = pd.read_csv(file_path)
descriptions = df[description_col].tolist()
corresponding_experts_names = df[corresponding_expert_col].tolist()

expertise_db = ExpertiseChromaDB()
expertise_dict = create_expertise_dict(descriptions, corresponding_experts_names)
expertise_db.add_expertise(expertise_dict)

In [3]:
plfss_json_path = f"{DATA_FOLDER}/{sheet_name}.json"
plfss_loader = PLFSSJsonLoader(plfss_json_path)

In [ ]:
filtered_df = plfss_loader.filter_amendements()
filtered_df["Expert conseillé"] = ""
filtered_df["Description"] = ""

for i in range(len(filtered_df)):
    line = filtered_df.iloc[i]
    exposé_des_motifs = line["Exposé des motifs"]
    results = expertise_db.query_documents(exposé_des_motifs, 1)
    expert_conseille = results["metadatas"][0][0]["experts"]
    description = results["metadatas"][0][0]["description"]
    filtered_df.at[filtered_df.index[i], "Expert conseillé"] = expert_conseille
    filtered_df.at[filtered_df.index[i], "Description"] = description

In [ ]:
output_path = f"{DATA_FOLDER}/attributed_plfss_2024.csv"
filtered_df = filtered_df.apply(
    lambda x: x.str.replace("\n", " ") if x.dtype == "object" else x
)
filtered_df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [ ]:
query_description = "Cet amendement vise à améliorer la vie des pompiers"
results = expertise_db.query_documents(query_description=query_description, n_results=3)

# Print the results
for doc_id, score, metadata in zip(
    results["ids"][0], results["distances"][0], results["metadatas"][0]
):
    print(
        f"Document ID: {doc_id}, Score: {score}, Description: {metadata['description']}, Expert: {metadata['experts']}"
    )